# Тетрадь 1.2: PDF — генерация и парсинг

Заявки на кредит приходят в банк не в виде JSON. Они приходят как **документы**: PDF, сканы, заполненные формы. Extraction-агент в фазе 2 должен уметь читать PDF и вытаскивать из него поля.

В этой тетради мы строим две стороны одного процесса:

1. **Генерация** — делаем PDF из объекта `Application`.
2. **Чтение** — читаем PDF и получаем обратно текст.

Обе стороны нужны: генерация — чтобы создавать тестовые заявки, чтение — чтобы работал агент.

**Чему ты научишься:**

- создавать PDF программно через `reportlab`;
- регистрировать шрифты с поддержкой кириллицы;
- читать PDF через `pdfplumber`;
- писать тест на round-trip (сгенерировал → прочитал → проверил);
- замечать типичные ловушки: неправильный шрифт, обрезанные строки, потерянные числа.

**Что на выходе:**

- `src/underwriting/pdf.py` с двумя функциями и одной константой.
- `tests/test_pdf.py` с тестом round-trip и негативным тестом.
- `data/pdf/` с 10 сгенерированными PDF-заявками.

**Пункты чеклиста:**

- №11: парсеры данных — теперь PDF.
- №1: Python-компоненты — чистые функции, работа с байтами.
- №18: отладка — научимся ловить ошибки в генерации и чтении.

**Формат работы:**

Каждая функция — отдельный раздел. Внутри: назначение, сигнатура, что делает по пунктам, готовый код, 1–2 задачи. Решения задач не даются.

## 0. Связь с предыдущими тетрадями

В фазе 0 у нас появился класс `Application` в `src/underwriting/schemas.py`. В тетради 1.3 (позже) мы будем генерировать 100 таких заявок и сохранять их в JSONL. В этой тетради мы возьмём одну заявку и превратим её в PDF.

В тетради 1.1 мы научились парсить HTML. PDF — это второй формат, с которым работает extraction-агент. Логика та же: сначала получить сырой текст, потом искать в нём структуру.

**Напоминание:** схема `Application` содержит поля `application_id`, `full_name`, `age`, `monthly_income`, `monthly_debt`, `loan_amount`, `loan_term_months`, `loan_purpose`, `employment_type`, `credit_history`, `region`, `applied_at`. Все они должны попасть в PDF.

In [1]:
# Импорты из фазы 0.
from datetime import date
from pathlib import Path

from underwriting.schemas import (
    Application,
    CreditHistory,
    EmploymentType,
    LoanPurpose,
    Region,
)

# Создадим тестовую заявку, с которой будем работать всю тетрадь.
example = Application(
    application_id="test-pdf-001",
    full_name="Иванов Иван Иванович",
    age=35,
    monthly_income=80_000,
    monthly_debt=5_000,
    loan_amount=500_000,
    loan_term_months=36,
    loan_purpose=LoanPurpose.CONSUMER,
    employment_type=EmploymentType.EMPLOYED,
    credit_history=CreditHistory.CLEAN,
    region=Region.MOSCOW,
    applied_at=date(2026, 10, 7),
)

print(example.model_dump_json(indent=2))

{
  "application_id": "test-pdf-001",
  "full_name": "Иванов Иван Иванович",
  "age": 35,
  "region": "moscow",
  "monthly_income": 80000.0,
  "monthly_debt": 5000.0,
  "loan_amount": 500000.0,
  "loan_term_months": 36,
  "loan_purpose": "consumer",
  "employment_type": "employed",
  "credit_history": "clean",
  "applied_at": "2026-10-07"
}


## 1. Зачем PDF и почему нельзя просто JSON

Можно задать вопрос: зачем усложнять, если у нас уже есть JSON? Ответ простой: **в реальной жизни заявки не приходят в JSON**.

- Клиент заполняет форму в отделении → система банка печатает PDF.
- Клиент скачивает PDF-анкету с сайта → заполняет → отправляет обратно.
- Клиент присылает скан паспорта и справку о доходах → PDF-скан.

Extraction-агент должен уметь работать с этими форматами. Мы начинаем с программно-сгенерированного PDF — он проще всего, потому что **мы сами знаем структуру**. Потом, если будет время, можно добавить OCR для сканов. Но даже без OCR 80% реальных заявок приходят как текстовый PDF.

**Разница с HTML-парсингом:** в HTML мы искали теги. В PDF тегов нет — есть только текст, координаты и порядок. Поэтому extraction-агент в фазе 2 будет получать уже очищенный текст из PDF и вытаскивать поля через промпт LLM.

## 2. Функция `application_to_pdf`: генерация PDF из заявки

### Что делает

Берёт объект `Application`, возвращает байты PDF-файла. Байты, а не путь к файлу, потому что в фазе 3 сервис будет отдавать PDF прямо в HTTP-ответ, не сохраняя на диск.

### Сигнатура

```python
def application_to_pdf(application: Application) -> bytes: ...
```

### По пунктам

1. Создаёт буфер в памяти (`io.BytesIO`) — не пишет сразу в файл.
2. Создаёт PDF-документ формата A4 через `reportlab.pdfgen.canvas.Canvas`.
3. Устанавливает шрифт **Source Serif Pro** (или его fallback) — обязательно с поддержкой кириллицы. Стандартный `Helvetica` русские буквы не отрисует.
4. Рисует заголовок «Кредитная заявка» в верхней части страницы.
5. Рисует поля заявки одно за другим: ID, ФИО, возраст, доход, долг, сумма, срок, цель, занятость, кредитная история, регион, дата подачи.
6. Для каждого поля форматирует значение: числа с разделителем тысяч, enum — через `.value`, дату — в формате `DD.MM.YYYY`.
7. Сохраняет PDF в буфер и возвращает байты.

### Где взять шрифт

В проекте Miya шрифт уже лежит в `backend/app/fonts/SourceSerifPro-Regular.ttf`. Скопируй его в наш проект:

```bash
mkdir -p assets/fonts
cp ~/Developer/Miya/backend/app/fonts/SourceSerifPro-Regular.ttf assets/fonts/
cp ~/Developer/Miya/backend/app/fonts/SourceSerifPro-Bold.ttf assets/fonts/
```

Если Miya нет под рукой — скачай Source Serif Pro с Google Fonts.

### Готовый код (скопируй в `src/underwriting/pdf.py`)

Создай файл `src/underwriting/pdf.py` вручную и вставь туда следующее. Я даю его целиком, потому что он важен и в нём каждая строчка на своём месте.

In [4]:
# Содержимое src/underwriting/pdf.py — скопируй вручную в файл.

"""
PDF-генерация и парсинг кредитных заявок.

Используется в extraction-агенте (фаза 2) для чтения PDF и в генераторе
тестовых данных (фаза 1.3) для создания документов.
"""
import io
import logging
from pathlib import Path

import pdfplumber
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfgen import canvas

from underwriting.schemas import Application

logger = logging.getLogger(__name__)

# Путь к шрифту. В проекте Miya этот же шрифт лежит в backend/app/fonts/.
FONT_DIR = Path(__file__).resolve().parents[2] / "assets" / "fonts"
FONT_REGULAR = FONT_DIR / "SourceSerifPro-Regular.ttf"
FONT_BOLD = FONT_DIR / "SourceSerifPro-Bold.ttf"


def _register_fonts() -> tuple[str, str]:
    """Зарегистрировать шрифты с поддержкой кириллицы. Вернуть (regular, bold)."""
    if not FONT_REGULAR.exists():
        logger.warning("Шрифт не найден: %s. Использую Helvetica (кириллица сломается).", FONT_REGULAR)
        return "Helvetica", "Helvetica-Bold"

    pdfmetrics.registerFont(TTFont("SourceSerifPro", str(FONT_REGULAR)))
    if FONT_BOLD.exists():
        pdfmetrics.registerFont(TTFont("SourceSerifPro-Bold", str(FONT_BOLD)))
    else:
        pdfmetrics.registerFont(TTFont("SourceSerifPro-Bold", str(FONT_REGULAR)))
    return "SourceSerifPro", "SourceSerifPro-Bold"


def application_to_pdf(application: Application) -> bytes:
    """Превратить объект Application в PDF-документ (байты)."""
    font_regular, font_bold = _register_fonts()

    buffer = io.BytesIO()
    c = canvas.Canvas(buffer, pagesize=A4)
    width, height = A4

    y = height - 72
    c.setFont(font_bold, 20)
    c.drawString(72, y, "Кредитная заявка")
    y -= 32

    c.setFont(font_regular, 11)
    c.drawString(72, y, f"Номер заявки: {application.application_id}")
    y -= 16

    fields = [
        ("ФИО", application.full_name),
        ("Возраст", f"{application.age} лет"),
        ("Месячный доход", f"{application.monthly_income:,.2f} ₽"),
        ("Существующие платежи", f"{application.monthly_debt:,.2f} ₽"),
        ("Сумма кредита", f"{application.loan_amount:,.2f} ₽"),
        ("Срок", f"{application.loan_term_months} месяцев"),
        ("Цель кредита", application.loan_purpose.value),
        ("Занятость", application.employment_type.value),
        ("Кредитная история", application.credit_history.value),
        ("Регион", application.region.value),
        ("Дата подачи", application.applied_at.strftime("%d.%m.%Y")),
    ]

    for label, value in fields:
        c.setFont(font_bold, 11)
        c.drawString(72, y, f"{label}:")
        c.setFont(font_regular, 11)
        c.drawString(260, y, str(value))
        y -= 18

    c.save()
    buffer.seek(0)
    return buffer.read()


def extract_text_from_pdf(pdf_bytes: bytes) -> str:
    """Извлечь весь текст из PDF (все страницы, склеенные через \\n)."""
    pages_text: list[str] = []
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages_text.append(text)
    return "\n".join(pages_text)


NameError: name '__file__' is not defined

### Проверка в тетради

После того как ты создашь `src/underwriting/pdf.py`, импортируй его сюда и сгенерируй первый PDF.

In [3]:
from underwriting.pdf import application_to_pdf, extract_text_from_pdf

pdf_bytes = application_to_pdf(example)
print(f"Размер PDF: {len(pdf_bytes)} байт")
print(f"Начинается с %PDF: {pdf_bytes[:4] == b'%PDF'}")

text = extract_text_from_pdf(pdf_bytes)
print()
print("Текст, извлечённый из PDF:")
print(text[:500])

Размер PDF: 26197 байт
Начинается с %PDF: True

Текст, извлечённый из PDF:
Кредитная заявка
Номер заявки: test-pdf-001
ФИО: Иванов Иван Иванович
Возраст: 35 лет
Месячный доход: 80,000.00 ₽
Существующие платежи: 5,000.00 ₽
Сумма кредита: 500,000.00 ₽
Срок: 36 месяцев
Цель кредита: consumer
Занятость: employed
Кредитная история: clean
Регион: moscow
Дата подачи: 07.10.2026


### Задача 2.1: проверь все поля

Сгенерируй PDF для заявки `example`. Прочитай текст обратно. Проверь, что в тексте есть:

- номер заявки `test-pdf-001`;
- ФИО `Иванов Иван Иванович`;
- сумма `500,000.00` (с запятой как разделителем тысяч);
- срок `36 месяцев`.

Если что-то пропало — подумай, почему. Может быть, слишком мелкий шрифт? Может, координаты не туда попали?

In [5]:
pdf = application_to_pdf(example)
text = extract_text_from_pdf(pdf)
print(text)

Кредитная заявка
Номер заявки: test-pdf-001
ФИО: Иванов Иван Иванович
Возраст: 35 лет
Месячный доход: 80,000.00 ₽
Существующие платежи: 5,000.00 ₽
Сумма кредита: 500,000.00 ₽
Срок: 36 месяцев
Цель кредита: consumer
Занятость: employed
Кредитная история: clean
Регион: moscow
Дата подачи: 07.10.2026


### Задача 2.2: сгенерируй PDF для другой заявки

Создай новый объект `Application` с другими полями (например, заявка на ипотеку, доход 150 000, сумма 3 000 000, срок 240 месяцев). Сгенерируй PDF, прочитай текст, убедись, что все поля на месте.

**Подсказка:** используй `LoanPurpose.MORTGAGE`, `EmploymentType.SELF_EMPLOYED`.

In [ ]:
# ТВОЙ КОД ЗДЕСЬ


## 3. Функция `extract_text_from_pdf`: чтение PDF

### Что делает

Берёт байты PDF, возвращает весь текст со всех страниц, склеенный через `\n`. Не пытается разобрать структуру — только достаёт текст.

### Сигнатура

```python
def extract_text_from_pdf(pdf_bytes: bytes) -> str: ...
```

### По пунктам

1. Открывает PDF из байтов через `pdfplumber.open(io.BytesIO(pdf_bytes))` — **не с диска**, а прямо из памяти.
2. Идёт по страницам через `pdf.pages`.
3. Для каждой страницы вызывает `page.extract_text()`. Он возвращает `None`, если на странице нет текста — это норма для сканов. Поэтому `or ""`.
4. Склеивает текст страниц через `\n`.

### Почему не с диска

В фазе 3 сервис будет получать PDF как загруженный файл. Файл придёт как байты в памяти. Писать его на диск, чтобы потом прочитать — лишний шаг. Функция принимает байты и возвращает строку. Так же, как `application_to_pdf` — наоборот.

### Ловушки, о которых надо знать

- **Пустой текст.** Сканированные PDF не содержат текста, только картинки. `extract_text()` вернёт `None`. Для таких PDF нужен OCR (Tesseract + `pytesseract`). Мы это не делаем — работаем с текстовыми PDF.
- **Потерянные пробелы.** PDF хранит текст с координатами, а не с пробелами. Иногда `pdfplumber` склеивает слова без пробела. В нашем случае это не критично — LLM всё равно разберётся.
- **Порядок строк.** PDF не гарантирует порядок текста. Обычно он сверху вниз, но бывает иначе. Для простых одностраничных заявок это не проблема.

### Задача 3.1: прочитай PDF из файла

Сохрани PDF-заявку на диск в `data/pdf/example.pdf`, потом прочитай файл как байты (`Path.read_bytes()`), передай в `extract_text_from_pdf` и выведи текст.

**Подсказка:** результат должен совпасть с тем, что ты получил при чтении из памяти.

In [ ]:
# ТВОЙ КОД ЗДЕСЬ


### Задача 3.2: обработай пустой PDF

Создай пустой PDF-документ (без текста) через `reportlab` и попробуй прочитать его через `extract_text_from_pdf`. Функция должна вернуть пустую строку без падения.

**Подсказка:** пустой PDF — это `canvas.Canvas(buffer, pagesize=A4); c.save()` без единого `drawString`.

In [ ]:
# ТВОЙ КОД ЗДЕСЬ


## 4. Тесты в `tests/test_pdf.py`

Создай файл `tests/test_pdf.py` вручную. Внутри — четыре теста.

**Тест 1: PDF генерируется.** `application_to_pdf(example)` возвращает байты, начинающиеся с `%PDF`.

**Тест 2: round-trip.** Сгенерировали PDF → прочитали текст → в тексте есть `test-pdf-001`, `Иванов Иван Иванович`, `500,000.00`.

**Тест 3: пустой PDF не падает.** Генерируем пустой PDF напрямую через `canvas` (не через нашу функцию), читаем — должна быть пустая строка.

**Тест 4: шрифт поддерживает кириллицу.** Сгенерировали PDF с русским текстом, прочитали — русские буквы сохранились, а не превратились в `?????`.

In [ ]:
Создай файл tests/test_pdf.py вручную и вставь туда код ниже.

"""Тесты PDF-генерации и парсинга."""
from datetime import date

from underwriting.pdf import application_to_pdf, extract_text_from_pdf
from underwriting.schemas import (
    Application, CreditHistory, EmploymentType, LoanPurpose, Region,
)


def _make_application() -> Application:
    return Application(
        application_id="test-pdf-001",
        full_name="Иванов Иван Иванович",
        age=35,
        monthly_income=80_000,
        monthly_debt=5_000,
        loan_amount=500_000,
        loan_term_months=36,
        loan_purpose=LoanPurpose.CONSUMER,
        employment_type=EmploymentType.EMPLOYED,
        credit_history=CreditHistory.CLEAN,
        region=Region.MOSCOW,
        applied_at=date(2026, 10, 7),
    )


def test_pdf_generated():
    pdf_bytes = application_to_pdf(_make_application())
    assert pdf_bytes[:4] == b"%PDF"


def test_round_trip_contains_all_fields():
    pdf_bytes = application_to_pdf(_make_application())
    text = extract_text_from_pdf(pdf_bytes)
    assert "test-pdf-001" in text
    assert "Иванов Иван Иванович" in text
    assert "500,000.00" in text
    assert "36 месяцев" in text


def test_empty_pdf_returns_empty_text():
    import io
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas

    buf = io.BytesIO()
    c = canvas.Canvas(buf, pagesize=A4)
    c.save()
    empty_bytes = buf.getvalue()

    assert extract_text_from_pdf(empty_bytes) == ""


def test_cyrillic_is_preserved():
    pdf_bytes = application_to_pdf(_make_application())
    text = extract_text_from_pdf(pdf_bytes)
    # Если шрифт не поддерживает кириллицу — русские буквы пропадут или станут вопросительными знаками
    assert "Кредитная заявка" in text
    assert "?????" not in text

### Задача 4.1: запусти тесты

После того как создашь `tests/test_pdf.py`, выполни:

```bash
uv run pytest tests/test_pdf.py -v
```

Все 4 теста должны пройти. Если какой-то падает — покажи вывод, разберём.

In [ ]:
# Здесь ничего писать не надо — просто запусти команду в терминале.
# Результат должен быть: 4 passed.

## 5. Генерация 10 PDF-заявок в `data/pdf/`

Теперь, когда функции работают, сгенерируем 10 разных заявок и сохраним их как PDF. Эти файлы пойдут в extraction-агент в фазе 2.

### Что делать

1. В тетради 1.3 у нас есть `generate_batch(30)` — генератор заявок. Импортируй его.
2. Сгенерируй 10 заявок.
3. Для каждой вызови `application_to_pdf` и сохрани файл в `data/pdf/{application_id}.pdf`.
4. Выведи список сохранённых файлов.

### Задача 5.1: сгенерируй 10 PDF-заявок

Напиши код, который создаёт 10 заявок и сохраняет их как PDF.

**Подсказка:** папку создавай через `Path("data/pdf").mkdir(parents=True, exist_ok=True)`.

In [ ]:
# ТВОЙ КОД ЗДЕСЬ


### Задача 5.2: проверь round-trip на всех 10

Пройди по всем сохранённым PDF, прочитай каждый, убедись, что ID заявки из имени файла совпадает с ID в тексте.

**Подсказка:** `for pdf_path in Path("data/pdf").glob("*.pdf")`.

In [ ]:
# ТВОЙ КОД ЗДЕСЬ


## 6. Что закрыто по чеклисту

- **№11 (парсеры сайтов и данных):** закрыт полностью — ты теперь умеешь парсить и HTML, и PDF.
- **№1 (Python-компоненты):** две чистые функции без побочных эффектов, работа с байтами в памяти.
- **№9 (логирование, обработка ошибок):** `logger.warning` при отсутствии шрифта.
- **№18 (отладка):** round-trip тест — основной инструмент отладки формата.

**Что дальше:** тетрадь 1.3 — генерация 100 заявок, расширение `generate.py` регионом и реалистичными распределениями. Это подготовка к 1.4 (EDA) и 1.5 (gold standard).

## 7. Напоминание: сквозные правила проекта

Всё, что мы делаем в этой тетради, следует трём принципам, которые уже проходили:

**1. Код живёт в `src/`, тетрадь импортирует.** Как `credit_math.py` из фазы 0 — не пиши функцию дважды. Если нужна правка — правь в модуле.

**2. Функции — чистые.** `application_to_pdf` не пишет в файл, не читает с диска, не знает про существование других систем. Вход → выход.

**3. Тесты обязательны.** Каждый новый модуль в `src/` покрывается тестом в `tests/`. Round-trip — самый важный тест для форматов: сгенерировал → прочитал → сравнил.

Держи это в голове — в фазе 3, когда будем оборачивать функции в FastAPI, ты уже не будешь их трогать. Ты просто позовёшь `application_to_pdf` из эндпоинта и вернёшь PDF-байты как HTTP-ответ.